# Azure Observable RAG — Pipeline Demo

Walks one PDF through the four ingestion + retrieval stages and renders the intermediate object at every step:

| § | Stage | Module |
|---|---|---|
| 1 | **Extract** | `src.extract` (live Document Intelligence call) |
| 2 | **Chunk**   | `src.chunk`   (Markdown-header + token-bounded split) |
| 3 | **Index**   | `src.embed` + `src.index` + `src.ingest.chunk_to_search_doc` (upsert into `kb-chunks`) |
| 4 | **Search**  | `src.search.hybrid_search` (BM25 + HNSW + L2 semantic) |

**Prereqs.** `.env` populated with Document Intelligence, Azure OpenAI (embed deployment), and AI Search credentials — typically by running `bash infra/deploy.sh`.

In [1]:
import json, sys, time
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from dotenv import load_dotenv
load_dotenv(REPO_ROOT / '.env')

import pandas as pd
from IPython.display import Markdown

from src.extract import extract
from src.chunk import chunk_document
from src.embed import embed_batch
from src.index import create_or_update_index, get_search_client
from src.ingest import chunk_to_search_doc
from src.search import hybrid_search

## 0 · Sample input

We use the Meraki MX67 installation guide — small, well-structured, exercises both DI markdown and figures.

In [2]:
LOCAL_PDF  = REPO_ROOT / 'data/devices/network_access/meraki_mx67/manuals/MX67_MX68 Installation Guide.pdf'
BLOB_PATH  = 'devices/network_access/meraki_mx67/manuals/MX67_MX68 Installation Guide.pdf'

print(f'file:  {LOCAL_PDF.name}')
print(f'size:  {LOCAL_PDF.stat().st_size / 1024:.1f} KB')
print(f'blob:  {BLOB_PATH}')

file:  MX67_MX68 Installation Guide.pdf
size:  299.0 KB
blob:  devices/network_access/meraki_mx67/manuals/MX67_MX68 Installation Guide.pdf


## 1 · Extract — `src.extract.extract`

Calls Document Intelligence (`prebuilt-layout`) on the PDF. Returns `ExtractedDocument` with markdown, structured tables, figure bounding boxes, and metadata derived from the blob path.

In [3]:
doc = extract(LOCAL_PDF, BLOB_PATH)

print(f'page_count : {doc.page_count}')
print(f'tables     : {len(doc.tables)}')
print(f'figures    : {len(doc.figures)}')
print(f'markdown   : {len(doc.markdown):,} chars')

page_count : 13
tables     : 4
figures    : 8
markdown   : 15,005 chars


**Path-derived metadata** — parsed deterministically from the blob path by `extract._parse_path_metadata`. These fields end up on every chunk and become filterable columns in the AI Search index.

In [4]:
pd.DataFrame([{
    'scope':         doc.scope,
    'device_family': doc.device_family,
    'device':        doc.device,
    'doc_type':      doc.doc_type,
    'topic':         doc.topic,
    'version':       doc.version,
    'is_shared':     doc.is_shared,
    'file_type':     doc.file_type,
}])

,scope,device_family,device,doc_type,topic,version,is_shared,file_type
0,device,network_access,meraki_mx67,manual,mx67-mx68-installation-guide,None,False,pdf


**Markdown preview** — first ~1500 chars of `doc.markdown`. Note the `<!-- PageNumber=N -->` page markers that `chunk.py` will use to attribute each chunk to a source page.

In [5]:
Markdown(doc.markdown[:1500] + ('\n\n_…truncated_' if len(doc.markdown) > 1500 else ''))

CISCO

Meraki


# MX67/MX68 Installation Guide

This document describes how to install and set up the MX67 and MX68 security appliance.
Additional reference documents are available online at: www.meraki.com/library/products.


# MX67/MX68 Overview

The Meraki MX67 and MX68 are enterprise security appliances designed for distributed
deployments that require remote administration. It is ideal for network administrators who
demand both ease of deployment and a state-of-the-art feature set. A full overview of the
appliances' features can be found in the MX67 and MX68 Overview and Specifications.


# Package Contents

In addition to the MX device, the following are provided:

MX67/MX68

MX67W/MX68W

Power Adapter (No Power Cable)

Power Adapter (No Power Cable)

2x CAT5e Ethernet Cables

2x CAT5e Ethernet Cables

2x WiFi Antennae


<table>
<tr>
<th></th>
<th></th>
</tr>
<tr>
<th>MX67C</th>
<th>MX68CW</th>
</tr>
</table>


Power Adapter (No Power Cable)

Power Adapter (No Power Cable)

2x CAT5e Ethernet Cables

2x CAT5e Ethernet Cables

2x LTE Antennae

2x Attached (Non-Removeable) Hybrid WiFi+LTE Antennae


# Front Panels

MX67/67C/67W

<!-- PageNumber="1" -->
<!-- PageBreak -->


# MX68/68W/68CW

\-


## Status Indicator

The MX67/MX68 series devices uses an LED to inform the user of the device's status. LED patterns and their meanings are described below.


<table>
<tr>
<td>LED Status</td>
<td>Meaning</td>
</tr>
<tr>
<td>Solid orange</td>
<td>Power is applied but the appliance i

_…truncated_

In [6]:
if doc.tables:
    tables_df = pd.DataFrame([
        {'page': t.page, 'columns': len(t.headers), 'rows': len(t.rows), 'headers': t.headers}
        for t in doc.tables[:5]
    ])
    display(tables_df)
else:
    print('no tables extracted')

,page,columns,rows,headers
0,1,2,1,"[, ]"
1,2,2,6,"[Field, Value]"
2,12,3,1,"[Product, Warranty Period, Warranty Information]"
3,13,3,2,"[Product, Warranty Period, Warranty Information]"


In [7]:
if doc.figures:
    figures_df = pd.DataFrame([
        {'figure_id': f.figure_id, 'page': f.page, 'polygon': [round(x, 2) for x in f.polygon]}
        for f in doc.figures[:5]
    ])
    display(figures_df)
else:
    print('no figures extracted')

,figure_id,page,polygon
0,p2_f1,2,"[0.55, 8.45, 7.9, 8.45, 7.91, 9.36, 0.55, 9.36]"
1,p3_f1,3,"[0.56, 1.36, 7.92, 1.36, 7.92, 5.85, 0.56, 5.84]"
2,p4_f1,4,"[0.57, 0.71, 7.9, 0.71, 7.9, 4.24, 0.57, 4.23]"
3,p5_f1,5,"[0.49, 3.09, 7.93, 3.09, 7.92, 6.14, 0.49, 6.14]"
4,p6_f1,6,"[0.56, 0.7, 7.92, 0.7, 7.92, 4.68, 0.56, 4.67]"


## 2 · Chunk — `src.chunk.chunk_document`

Two-stage split: by Markdown headers first (so each chunk carries its `heading_path`), then token-bounded via `tiktoken` with **`CHUNK_MAX_TOKENS=512`** and **`CHUNK_OVERLAP_TOKENS=50`** (~10% overlap, both env-overridable). Each chunk gets a deterministic content-hashed `chunk_id` so re-ingestion is idempotent.

In [8]:
chunks = chunk_document(doc)

lengths = [len(c.content) for c in chunks]
print(f'chunks         : {len(chunks)}')
print(f'content (chars): mean {sum(lengths)//len(lengths)}  min {min(lengths)}  max {max(lengths)}')
print(f'distinct heading paths: {len({c.heading_path for c in chunks})}')

chunks         : 31
content (chars): mean 466  min 13  max 2145
distinct heading paths: 30


In [ ]:
chunks_df = pd.DataFrame([{
    'idx':          c.chunk_index,
    'chunk_id':     c.chunk_id[:12] + '…',
    'page':         c.page_number,
    'heading_path': c.heading_path,
    'preview':      c.content[:120].replace('\n', ' ') + ('…' if len(c.content) > 120 else ''),
} for c in chunks])
chunks_df

In [13]:
per_page = pd.Series([c.page_number for c in chunks]).value_counts().sort_index()
per_page.rename_axis('page').to_frame('chunks')

,chunks
page,
1,5
2,3
4,1
6,1
7,1
8,2
9,3
10,5
11,4


## 3 · Index — embed + upsert into `kb-chunks`

Three steps:
1. **Ensure the index exists** (`create_or_update_index` is idempotent).
2. **Embed each chunk's content** via `text-embedding-3-small` (1536-dim).
3. **Shape into search docs** (using `chunk_to_search_doc` — the same helper the Function App's blob trigger uses) and call `upload_documents`.

Idempotent: `chunk_id` is content-hashed, so re-running this cell is a semantic no-op.

In [14]:
index_name = create_or_update_index()
search_client = get_search_client()
doc_count_before = search_client.get_document_count()
print(f'index            : {index_name}')
print(f'docs before this : {doc_count_before:,}')

index            : kb-chunks
docs before this : 852


In [15]:
vectors = embed_batch([c.content for c in chunks])
print(f'vectors        : {len(vectors)}')
print(f'embedding dim  : {len(vectors[0])}')
print(f'preview [0][:5]: {[round(x, 4) for x in vectors[0][:5]]}')

vectors        : 31
embedding dim  : 1536
preview [0][:5]: [0.0303, -0.0351, -0.0056, 0.011, 0.007]


**First search-doc as JSON** — this is the exact shape that goes into AI Search. `tables_json` / `figures_json` are stringified because the index schema flattens them; `content_vector` carries the embedding.

In [16]:
search_docs = [chunk_to_search_doc(c, v) for c, v in zip(chunks, vectors)]
preview = dict(search_docs[0])
preview['content_vector'] = f'<{len(preview["content_vector"])}-d float[]>'
preview['content']        = preview['content'][:200] + ('…' if len(preview['content']) > 200 else '')
print(json.dumps(preview, indent=2, ensure_ascii=False))

{
  "chunk_id": "32e74e4eba121c1ccab7a601633dc5222f5f322e9f212db70aee52d1c4688dfd",
  "chunk_index": 0,
  "content": "CISCO\n\nMeraki",
  "doc_id": "b97e95bd5b073463",
  "source_path": "devices/network_access/meraki_mx67/manuals/MX67_MX68 Installation Guide.pdf",
  "file_name": "MX67_MX68 Installation Guide.pdf",
  "file_type": "pdf",
  "category": "manual",
  "page_number": 1,
  "page_end": 1,
  "heading_path": null,
  "scope": "device",
  "device_family": "network_access",
  "device": "meraki_mx67",
  "doc_type": "manual",
  "topic": "mx67-mx68-installation-guide",
  "version": null,
  "is_shared": false,
  "tables_json": "",
  "figures_json": "",
  "content_vector": "<1536-d float[]>"
}


In [17]:
result = search_client.upload_documents(search_docs)
succeeded = sum(1 for r in result if r.succeeded)
print(f'upserted: {succeeded}/{len(search_docs)}')

time.sleep(2)  # AI Search commit lag
doc_count_after = search_client.get_document_count()
print(f'docs after : {doc_count_after:,}  (delta: {doc_count_after - doc_count_before:+d})')

upserted: 31/31
docs after : 852  (delta: +0)


## 4 · Search — `src.search.hybrid_search`

Pure retrieval — no LLM, no orchestration. `hybrid_semantic` mode runs BM25 + HNSW vector search in parallel, then applies Azure's L2 semantic reranker. This is exactly what the LangGraph DAG calls under the hood.

In [18]:
QUERY = 'How do I mount the Meraki MX67 to a wall?'
results = hybrid_search(QUERY, top_k=5, search_mode='hybrid_semantic')

indexed_ids = {c.chunk_id for c in chunks}
results_df = pd.DataFrame([{
    'rank':           r.rank,
    'score':          round(r.score, 4),
    'reranker_score': round(r.reranker_score, 4) if r.reranker_score is not None else None,
    'page':           r.page_number,
    'heading_path':   r.heading_path,
    'caption':        (r.semantic_caption or '')[:90],
    'chunk_id':       r.chunk_id[:12] + '…',
    'from_this_run':  r.chunk_id in indexed_ids,
} for r in results])
results_df

,rank,score,reranker_score,page,heading_path,caption,chunk_id,from_this_run
0,1,0.0182,2.7430,11,Product Overview > Others,## Others · Paper eject position are selectab...,ccd34f373dc3…,False
1,2,0.0313,2.6593,1,MX67/MX68 Installation Guide,# MX67/MX68 Installation Guide This document ...,4643575b6784…,True
2,3,0.0242,2.4558,9,Configure your Dashboard Network,# Configure your Dashboard Network The follow...,76aab519abee…,True
3,4,0.0238,2.3265,10,Check and Configure Upstream Firewall Settings...,## Mounting Hardware The supplied wall screws...,5f5d650c2b60…,True
4,5,0.0137,2.2215,13,Accessories > Options,## Options · Wireless LAN cable set (Model: O...,e79f7b116bba…,False


**Off-topic contrast.** Same `top_k`, same mode — but a query the document doesn't answer. Expect noticeably lower `reranker_score` values.

In [19]:
OFF_TOPIC = 'How do I configure a payment terminal pin pad?'
off = hybrid_search(OFF_TOPIC, top_k=5, search_mode='hybrid_semantic')
off_df = pd.DataFrame([{
    'rank':           r.rank,
    'reranker_score': round(r.reranker_score, 4) if r.reranker_score is not None else None,
    'file_name':      r.file_name,
    'page':           r.page_number,
    'heading_path':   r.heading_path,
} for r in off])
off_df

,rank,reranker_score,file_name,page,heading_path
0,1,2.2438,Desk5000 and 3000 series - User guide.pdf,8,Desk Series > CAUTION
1,2,2.2024,Desk5000 and 3000 series - User guide.pdf,8,Desk Series > CAUTION > 3.4 Desk series : Fixe...
2,3,2.0300,tm-m30ii_trg_en_reva.pdf,72,Controlling the Cash Drawer
3,4,2.0157,tm-m30ii_trg_en_reva.pdf,65,Restore Default Values Mode > 1 After running ...
4,5,1.9817,Desk5000 and 3000 series - User guide.pdf,11,Desk Series > 4.Installation > 4.1 Positioning...


## 5 · Recap

- **Extract** — [src/extract.py](../src/extract.py) → `ExtractedDocument(markdown, tables, figures, +path metadata)`
- **Chunk** — [src/chunk.py](../src/chunk.py) → `list[Chunk]` (header + token-bounded split, content-hashed IDs)
- **Index** — [src/embed.py](../src/embed.py) + [src/index.py](../src/index.py) + [src/ingest.py](../src/ingest.py)`.chunk_to_search_doc` → upsert into `kb-chunks`
- **Search** — [src/search.py](../src/search.py)`.hybrid_search` → `list[RetrievalResult]`

**Run the same pipeline over the whole `data/` corpus:** `python -m src.ingest`.

**See it end-to-end (with the LangGraph DAG + LLM generation):** `bash infra/start_chainlit.sh`.